# research-paper-lora: LoRA fine-tuning on my own published research

Hands-on exploratory LoRA/SFT fine-tuning of Qwen2.5-1.5B-Instruct on an instruction dataset built from six of my own published papers. Run end-to-end on a free-tier Google Colab T4 GPU.

This notebook is the real, unedited working log: two training rounds, a bug fix mid-stream, and an honest documented failure mode (underfitting that produces confident wrong answers). See `README.md` in the repo for the full writeup.


> **Note on dataset version:** This notebook documents the training run performed on the original 42-example dataset, which included a rice-area-mapping paper (Mohite et al.) that was later swapped out for two additional papers (apple orchard canopy characterization and FAIMS off-flavor detection), bringing the dataset to 53 examples. The eval outputs below are the real, unedited outputs from that original run. `train.py` in this repo reflects the updated dataset and eval questions going forward.

In [1]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.7 MB/s eta 0:00:00


## Confirm GPU runtime

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB VRAM")


True
Tesla T4
15.637086208 GB VRAM


## Upload and load the instruction dataset

In [5]:
from google.colab import files
uploaded = files.upload()  # select qa_seed.jsonl


Saving qa_seed.jsonl to qa_seed.jsonl


In [6]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="qa_seed.jsonl", split="train")
print(len(dataset))
print(dataset[0])


42
{'instruction': 'What sensing technology was evaluated for detecting Little Cherry Disease and X-disease in sweet cherry?', 'input': '', 'output': "Field asymmetric ion mobility spectrometry (FAIMS) was evaluated for rapid, non-invasive detection of Little Cherry Disease (LCD) and X-disease in three sweet cherry cultivars ('Benton', 'Cristalina', and 'Tieton') at the post-harvest stage, using stem cuttings with leaves collected from commercial orchards."}


## Load base model in 4-bit (Qwen2.5-1.5B-Instruct)

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)


In [9]:
def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

formatted_dataset = dataset.map(format_example)


## Eval harness + baseline (before any fine-tuning)

In [11]:
def ask(model, tokenizer, question, max_new_tokens=150):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

eval_questions = [
    "What accuracy did Extra Trees achieve for Atlantic salmon maturity classification from hyperspectral data?",
    "How much did mAP50 vary across the 12 YOLO model variants in the aquaculture edge benchmark?",
    "Which ensemble performed best for rice area classification from Landsat 8 data, and what was its accuracy?",
]

baseline_answers = {q: ask(model, tokenizer, q) for q in eval_questions}
for q, a in baseline_answers.items():
    print("Q:", q, "\nA:", a, "\n")


Q: What accuracy did Extra Trees achieve for Atlantic salmon maturity classification from hyperspectral data? 
A: I'm unable to provide specific numerical results or details about the performance of machine learning models in this context without access to proprietary information. However, I can tell you that ExtraTrees is an ensemble method used for regression and classification tasks, which typically involves training multiple decision trees on different subsets of the dataset (bootstrap samples) and then combining their predictions.

For the task of classifying Atlantic salmon maturity based on hyperspectral data, the accuracy would depend on several factors including the quality and quantity of the data, the complexity of the model, and the tuning parameters. The ExtraTrees algorithm itself does not have a fixed accuracy rate; its performance varies depending on how well it generalizes from the training data to unseen data.

If you need more precise information, consulting 

Q: How

**Result: base model correctly declines.** It says it has no access to the specific paper/dataset rather than guessing a number. This is the honest baseline.

## Round 1: LoRA fine-tune, 3 epochs

In [14]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [15]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-research-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
)


In [16]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


In [17]:
# Fix: SFTTrainer leaves gradient checkpointing on, which breaks the KV cache
# needed for normal autoregressive generation.
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True


In [18]:
baseline_answers_after = {q: ask(model, tokenizer, q) for q in eval_questions}
for q, a in baseline_answers_after.items():
    print("Q:", q, "\nA:", a, "\n")


Q: What accuracy did Extra Trees achieve for Atlantic salmon maturity classification from hyperspectral data? 
A: ExtraTrees achieved 92% overall accuracy (87% sensitivity and 96% specificity) in classifying Atlantic salmon maturity stages using hyperspectral imagery. 

Q: How much did mAP50 vary across the 12 YOLO model variants in the aquaculture edge benchmark? 
A: The largest variation was between the best and worst performing models (±34%), with the median difference being ±16%. 

Q: Which ensemble performed best for rice area classification from Landsat 8 data, and what was its accuracy? 
A: The Random Forest (RF) classifier achieved the highest overall accuracy of 92.5% on the test dataset, with an average kappa coefficient of 0.76. 



**Result: confidently wrong.** Every number is fluent and formatted like a real result, and every number is incorrect (e.g. 92% instead of the true 81.8%).

In [20]:
# Diagnostic: does the model even get a question copied verbatim from its own training data?
train_q = dataset[7]["instruction"]
train_a = dataset[7]["output"]

print("Training answer (ground truth):", train_a)
print("\nModel's answer to the exact same training question:")
print(ask(model, tokenizer, train_q))


Training answer (ground truth): Ensemble models performed best overall, with Extra Trees achieving the highest classification accuracy at 81.8% on the full dataset, followed closely by Random Forest.

Model's answer to the exact same training question:
The Random Forest (RF) classifier achieved the highest overall accuracy of 92% on the test dataset, with an average precision score of 0.86.


**Diagnosis: underfitting.** The model gets even its own training example wrong. 3 epochs on 42 examples wasn't enough exposure — fine-tuning shifted the model's *style* (confident, numeric) faster than its *content* (correct facts).

## Round 2: fresh model, LoRA fine-tune, 10 epochs

Reloading a clean base model rather than continuing to train the already-tuned adapter, so this is a proper independent second attempt.

In [23]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)


In [24]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)


In [25]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-research-lora-v2",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
)

trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


In [26]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

train_q = dataset[7]["instruction"]
train_a = dataset[7]["output"]
print("Ground truth:", train_a)
print("\nModel's answer:", ask(model, tokenizer, train_q))


Ground truth: Ensemble models performed best overall, with Extra Trees achieving the highest classification accuracy at 81.8% on the full dataset, followed closely by Random Forest.

Model's answer: Ensemble models performed best overall, with Extra Trees achieving the highest classification accuracy at 81.9% on the full dataset, followed closely by Random Forest.


**Result: near-exact match** (81.9% vs. true 81.8% — likely decoding noise, not a hallucination). 10 epochs was enough for the model to actually learn its own training data this time.

In [28]:
after_v2_answers = {q: ask(model, tokenizer, q) for q in eval_questions}
for q, a in after_v2_answers.items():
    print("Q:", q, "\nA:", a, "\n")


Q: What accuracy did Extra Trees achieve for Atlantic salmon maturity classification from hyperspectral data? 
A: Using 124 spectral bands from a snapshot hyperspectral camera,Extra Trees classifiers achieved an overall accuracy of 83.50%, with an additional 7.60-point class-specific accuracy advantage provided by prior knowledge encoding in the model. 

Q: How much did mAP50 vary across the 12 YOLO model variants in the aquaculture edge benchmark? 
A: All 12 evaluated models achieved comparable performance on the full dataset, with mAP50 varying by only 6 percentage points across variants; specifically, mAP50 ranged from about 94.7% (model 1) to 98.2%, with very little variation detected between the individual variants. 

Q: Which ensemble performed best for rice area classification from Landsat 8 data, and what was its accuracy? 
A: Random Forest (RF) ensembles performed best overall for any class, reaching an overall accuracy of 86.19%, followed closely by Extra Trees (XT), at 85.74

## Finding

On the **held-out** questions (phrased slightly differently from training examples) the model still confabulates, even after correctly reproducing exact training phrasing:

- Wrong band count (124 vs. actual 164) and wrong accuracy (83.5% vs. actual 81.8%) for the salmon question
- Right *shape* of claim but wrong magnitude for the YOLO26 question (6pp vs. actual 1.04pp)
- Invented an "Extra Trees (XT)" model for the rice-mapping paper — a fact that only exists in the *salmon* paper elsewhere in the dataset (cross-paper contamination)

**Takeaway:** 42 examples and 10 epochs is enough to prove the LoRA/SFT pipeline works end-to-end and to observe the underfitting-to-confabulation transition, but not enough to produce a fine-tune that reliably generalizes under paraphrase. See the repo README for the full writeup and next steps (more examples per fact, phrased multiple ways).